In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.stats import norm

# Load your data
df = pd.read_csv("purchase.csv")

X = df['idx']
y = df['purchase']


In [5]:
X_const = sm.add_constant(X)
glm_model = sm.GLM(y, X_const, family=sm.families.Binomial()).fit()
print(glm_model.summary())

# Confidence Interval for slope coefficient
print("GLM 95% CI (slope):", glm_model.conf_int().loc['idx'])


                 Generalized Linear Model Regression Results                  
Dep. Variable:               purchase   No. Observations:                 2000
Model:                            GLM   Df Residuals:                     1998
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -769.29
Date:                Fri, 02 May 2025   Deviance:                       1538.6
Time:                        13:53:58   Pearson chi2:                 2.03e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.06354
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.2196      0.160    -20.175      0.0

In [6]:
def log_likelihood(params):
    b0, b1 = params
    linear_pred = b0 + b1 * X
    p = 1 / (1 + np.exp(-linear_pred))
    return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

res = minimize(log_likelihood, [0, 0])
b0_hat, b1_hat = res.x
print("Manual MLE slope estimate:", b1_hat)

# Hessian approximation
eps = np.sqrt(np.finfo(float).eps)
hess_inv = res.hess_inv if hasattr(res, 'hess_inv') else res.hess_inv(np.eye(2))
se_b1 = np.sqrt(np.diag(hess_inv))[1]
z = norm.ppf(0.975)
ci_mle = (b1_hat - z * se_b1, b1_hat + z * se_b1)
print("MLE 95% CI (slope):", ci_mle)


Manual MLE slope estimate: 0.032526593189833704
MLE 95% CI (slope): (0.026745931913805844, 0.038307254465861564)


In [7]:
np.random.seed(42)
n_boot = 1000
boot_coefs = []

for _ in range(n_boot):
    sample = df.sample(frac=1, replace=True)
    Xb = sm.add_constant(sample['idx'])
    yb = sample['purchase']
    model = sm.GLM(yb, Xb, family=sm.families.Binomial()).fit()
    boot_coefs.append(model.params['idx'])

boot_coefs = np.array(boot_coefs)

# Standard deviation method
mean_boot = np.mean(boot_coefs)
se_boot = np.std(boot_coefs)
ci_sd = (mean_boot - z * se_boot, mean_boot + z * se_boot)
print("Bootstrap CI (Std Dev):", ci_sd)

# Quantile method
ci_quantile = (np.percentile(boot_coefs, 2.5), np.percentile(boot_coefs, 97.5))
print("Bootstrap CI (Quantile):", ci_quantile)


Bootstrap CI (Std Dev): (0.0268951923032092, 0.03831248770366256)
Bootstrap CI (Quantile): (0.027036336742558606, 0.038506335331263256)


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import statsmodels.api as sm

# Load data
df = pd.read_csv("purchase.csv")
X = df['idx'].values
y = df['purchase'].values
n = len(y)

# === Step 1: Likelihood Function ===
def neg_log_likelihood(params):
    b0, b1 = params
    linear = b0 + b1 * X
    p = 1 / (1 + np.exp(-linear))  # inverse logit
    log_lik = y * np.log(p) + (1 - y) * np.log(1 - p)
    return -np.sum(log_lik)  # return negative log-likelihood

# === Step 2: MLE Estimation ===
res = minimize(neg_log_likelihood, [0, 0], method='BFGS')
b0_hat, b1_hat = res.x
print("MLE Estimate (slope):", b1_hat)

# === Step 3: Frequentist Inference (Hessian for SE) ===
hess_inv = res.hess_inv  # inverse Hessian ≈ covariance matrix
se_b1 = np.sqrt(np.diag(hess_inv))[1]  # std error for slope
z = norm.ppf(0.975)
ci_mle = (b1_hat - z * se_b1, b1_hat + z * se_b1)
print("MLE 95% CI:", ci_mle)


MLE Estimate (slope): 0.032526593189833704
MLE 95% CI: (0.026745931913805844, 0.038307254465861564)


In [2]:
np.random.seed(42)
B = 1000
boot_slopes = []

for _ in range(B):
    sample = df.sample(n=n, replace=True)
    Xb = sample['idx'].values
    yb = sample['purchase'].values
    
    def boot_nll(params):
        b0, b1 = params
        lin = b0 + b1 * Xb
        p = 1 / (1 + np.exp(-lin))
        return -np.sum(yb * np.log(p) + (1 - yb) * np.log(1 - p))
    
    res_b = minimize(boot_nll, [0, 0], method='BFGS')
    if res_b.success:
        boot_slopes.append(res_b.x[1])  # only keep slope

boot_slopes = np.array(boot_slopes)

# CI using Std Dev approach
boot_mean = np.mean(boot_slopes)
boot_se = np.std(boot_slopes)
ci_boot_sd = (boot_mean - z * boot_se, boot_mean + z * boot_se)
print("Bootstrap CI (Std Dev):", ci_boot_sd)

# CI using Quantile approach
ci_boot_quant = (np.percentile(boot_slopes, 2.5), np.percentile(boot_slopes, 97.5))
print("Bootstrap CI (Quantile):", ci_boot_quant)


Bootstrap CI (Std Dev): (0.02691861762866039, 0.03708492580660172)
Bootstrap CI (Quantile): (0.027041927786135515, 0.03707209761670532)


In [4]:
import pandas as pd

# Load your data
df = pd.read_excel("UCSD MGTA 495 Spring 2025 - MaxDiff1 Design & choices.xlsx")  # replace with your actual file

# Count how many times each item was shown
shown_counts = df['Item'].value_counts().sort_index()

# Count how many times each item was chosen as "best" (response == 1)
best_counts = df[df['Response'] == 1]['Item'].value_counts().sort_index()

# Count how many times each item was chosen as "worst" (response == -1)
worst_counts = df[df['Response'] == -1]['Item'].value_counts().sort_index()

# Align indices and fill missing values with 0
best_counts = best_counts.reindex(shown_counts.index, fill_value=0)
worst_counts = worst_counts.reindex(shown_counts.index, fill_value=0)

# Calculate percentage scores
best_pct = best_counts / shown_counts
worst_pct = worst_counts / shown_counts

# Final MaxDiff Score = % Best - % Worst
scores = best_pct - worst_pct

# Create a final summary DataFrame
summary = pd.DataFrame({
    'Shown': shown_counts,
    '% Best': best_pct,
    '% Worst': worst_pct,
    'MaxDiff Score': scores
})

# Sort by score (optional)
summary_sorted = summary.sort_values(by='MaxDiff Score', ascending=False)

print(summary_sorted)

      Shown    % Best   % Worst  MaxDiff Score
Item                                          
8       103  0.563107  0.252427       0.310680
12      105  0.428571  0.161905       0.266667
5       104  0.394231  0.134615       0.259615
6       100  0.430000  0.200000       0.230000
2       101  0.227723  0.148515       0.079208
9       103  0.097087  0.155340      -0.058252
13      103  0.087379  0.155340      -0.067961
7       102  0.235294  0.313725      -0.078431
10      102  0.147059  0.245098      -0.098039
11      103  0.145631  0.300971      -0.155340
3        99  0.141414  0.313131      -0.171717
1       105  0.209524  0.390476      -0.180952
4       102  0.137255  0.480392      -0.343137
